In [1]:
import pandas as pd
import numpy as np
import glob
import os
import re

In [2]:
tennis_data_path = "datos_partidos_sackmann/"
# Crea una lista con todos los archivos de la ruta que acaban en .csv
files = glob.glob(os.path.join(tennis_data_path, "*.csv"))
years_list = []

for file in files:
    # Lee cada archivo de la lista
    year = pd.read_csv(file, encoding='latin1')
    # Todos las columnas en mayúsculas para evitar errores
    year.columns = year.columns.str.upper().str.strip()
    years_list.append(year)

In [3]:
len(years_list)

36

In [4]:
# Junta todos los torneos en un único dataframe
df = pd.concat(years_list, axis=0, ignore_index=True, sort=False)

# Ordena por fecha
clean_dates = (df['TOURNEY_DATE'].astype(str).str.replace(r'\.0$', '', regex=True))
df['TOURNEY_DATE'] = pd.to_datetime(clean_dates, format='%Y%m%d', errors='coerce')
df = df.dropna(subset=['TOURNEY_DATE']).sort_values('TOURNEY_DATE')

In [6]:
df

,TOURNEY_ID,TOURNEY_NAME,SURFACE,DRAW_SIZE,TOURNEY_LEVEL,TOURNEY_DATE,MATCH_NUM,WINNER_ID,WINNER_SEED,WINNER_ENTRY,...,L_1STIN,L_1STWON,L_2NDWON,L_SVGMS,L_BPSAVED,L_BPFACED,WINNER_RANK,WINNER_RANK_POINTS,LOSER_RANK,LOSER_RANK_POINTS
0,1991-339,Adelaide,Hard,32,A,1990-12-31,1,101723,NaN,NaN,...,62.0,44.0,23.0,16.0,6.0,8.0,56.0,NaN,2.0,NaN
33,1991-354,Wellington,Hard,32,A,1990-12-31,3,101432,NaN,Q,...,79.0,36.0,14.0,14.0,7.0,16.0,172.0,NaN,249.0,NaN
34,1991-354,Wellington,Hard,32,A,1990-12-31,4,101439,8.0,NaN,...,61.0,38.0,7.0,13.0,4.0,10.0,70.0,NaN,263.0,NaN
35,1991-354,Wellington,Hard,32,A,1990-12-31,5,101735,3.0,NaN,...,40.0,17.0,13.0,10.0,7.0,13.0,32.0,NaN,103.0,NaN
36,1991-354,Wellington,Hard,32,A,1990-12-31,6,101119,NaN,NaN,...,35.0,19.0,7.0,9.0,4.0,8.0,83.0,NaN,90.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112478,2026-0322,Geneva,Clay,28,A,2026-05-17,362,126774,NaN,WC,...,63.0,48.0,4.0,11.0,3.0,4.0,82.0,725.0,80.0,735.0
112479,2026-0322,Geneva,Clay,28,A,2026-05-17,361,126239,5.0,NaN,...,34.0,22.0,7.0,8.0,5.0,8.0,24.0,1711.0,58.0,887.0
112480,2026-0322,Geneva,Clay,28,A,2026-05-17,360,111513,NaN,WC,...,58.0,39.0,18.0,14.0,10.0,14.0,238.0,241.0,83.0,722.0
112483,2026-0414,Hamburg,Clay,32,A,2026-05-17,384,209860,NaN,Q,...,31.0,21.0,7.0,9.0,1.0,5.0,57.0,893.0,94.0,649.0


In [6]:
# Elimina columnas que no son de interes
df = df.drop(columns=['MATCH_NUM','WINNER_ENTRY', 'LOSER_ENTRY'])

# Modifica valores de interes
df = df[~df['TOURNEY_NAME'].str.contains("Davis Cup", case=False, na=False)]
df = df[~df['TOURNEY_NAME'].str.contains("Olympics", case=False, na=False)]
df = df[~df['TOURNEY_NAME'].str.contains("Grand Slam Cup", case=False, na=False)]
df = df[~df['TOURNEY_NAME'].str.contains("Laver Cup", case=False, na=False)]
df['TOURNEY_NAME'] = df['TOURNEY_NAME'].replace("'s Hertogenbosch", "s-Hertogenbosch")
df['TOURNEY_NAME'] = df['TOURNEY_NAME'].replace("'s-Hertogenbosch", "s-Hertogenbosch")
df['TOURNEY_NAME'] = df['TOURNEY_NAME'].replace("Belgrade ", "Belgrade")
df['TOURNEY_NAME'] = df['TOURNEY_NAME'].replace("NextGen Finals", "Next Gen Finals")
df['TOURNEY_NAME'] = df['TOURNEY_NAME'].replace("Rio De Janeiro", "Rio de Janeiro")
df['TOURNEY_NAME'] = df['TOURNEY_NAME'].replace("St Petersburg", "St. Petersburg")

In [7]:
def separate_score_sets(score):
    # Comprueba si el valor es nulo o no es texto
    if pd.isna(score) or not isinstance(score, str):
        return [np.nan] * 10
    
    # Eliminamos cualquier número dentro de paréntesis (puntos tie-break)
    clean_score = re.sub(r'\(.*?\)', '', score)
    
    # Separamos el resultado en sets por espacios
    sets = clean_score.split()
    
    results = []
    
    for i in range(5):
        if i < len(sets):
            # Dividimos el set por el guion
            games = sets[i].split('-')
            if len(games) == 2:
                try:
                    results.append(float(games[0]))
                    results.append(float(games[1]))
                except ValueError:
                    # En caso de letras extrañas (como "RET" o "W/O"), devolvemos nulos
                    results.extend([np.nan, np.nan])
            else:
                results.extend([np.nan, np.nan])
        else:
            # Si el partido acabó en 2, 3 ó 4 sets, rellenamos el resto con nulos
            results.extend([np.nan, np.nan])
            
    return results

# 4. Aplicar la función y crear las 10 columnas
columns_names = ['W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5']

# Usamos .apply para generar una Serie de listas y expandirlas en columnas
df[columns_names] = pd.DataFrame(df['SCORE'].apply(separate_score_sets).tolist(), index=df.index)

print("Columnas de sets creadas correctamente.")

Columnas de sets creadas correctamente.


In [8]:
df

,TOURNEY_ID,TOURNEY_NAME,SURFACE,DRAW_SIZE,TOURNEY_LEVEL,TOURNEY_DATE,WINNER_ID,WINNER_SEED,WINNER_NAME,WINNER_HAND,...,W1,L1,W2,L2,W3,L3,W4,L4,W5,L5
0,1991-339,Adelaide,Hard,32,A,1990-12-31,101723,NaN,Magnus Larsson,R,...,6.0,4.0,3.0,6.0,7.0,6.0,NaN,NaN,NaN,NaN
33,1991-354,Wellington,Hard,32,A,1990-12-31,101432,NaN,Dimitri Poliakov,R,...,6.0,2.0,4.0,6.0,6.0,4.0,NaN,NaN,NaN,NaN
34,1991-354,Wellington,Hard,32,A,1990-12-31,101439,8.0,Javier Sanchez,R,...,6.0,3.0,3.0,6.0,6.0,3.0,NaN,NaN,NaN,NaN
35,1991-354,Wellington,Hard,32,A,1990-12-31,101735,3.0,Richard Fromberg,R,...,7.0,5.0,6.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
36,1991-354,Wellington,Hard,32,A,1990-12-31,101119,NaN,Marian Vajda,R,...,6.0,4.0,6.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112478,2026-0322,Geneva,Clay,28,A,2026-05-17,126774,NaN,Stefanos Tsitsipas,R,...,6.0,4.0,7.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
112479,2026-0322,Geneva,Clay,28,A,2026-05-17,126239,5.0,Arthur Rinderknech,R,...,6.0,4.0,6.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
112480,2026-0322,Geneva,Clay,28,A,2026-05-17,111513,NaN,Laslo Djere,R,...,6.0,4.0,3.0,6.0,6.0,3.0,NaN,NaN,NaN,NaN
112483,2026-0414,Hamburg,Clay,32,A,2026-05-17,209860,NaN,Ignacio Buse,R,...,6.0,1.0,6.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df.to_csv('sackmann.csv', index=False)